# Análisis Operaciones — Alojamientos Turísticos
### Desafío Nº 1 · Equip 29 · Perfil: **Analista de Operaciones y Gestión de Inventario**

**Contexto del negocio:** cada fila es un anuncio de un apartamento en alquiler turístico. La **disponibilidad** (`availability_30/60/90/365`) la fija el *host* (dueño), e indica los días libres en cada horizonte. La variable `insert_date` es la **fecha de extracción** de la información desde otro software, por lo que el dataset es una *fotografía* (no una serie temporal completa).

**Enfoque de operaciones e inventario:** el foco está en la **disponibilidad y la ocupación estimada** como factor de la presión de la demanda y la capacidad de oferta.

**Pregunta de negocio:** ¿Cual es la disponibilidad media de los alojamientos turisticos en los diferentes slost temporales (30, 60, 90 y 365 días) en cada ciudad?

## 00.- Librerias

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

%matplotlib inline
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 160)
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (9, 5)

## 01.- Cargar dataset limpio

In [2]:
df_limpio = pd.read_parquet("2026_05_25_pisos_turisticos_limpio.parquet")
df_limpio

,apartment_id,name,description,host_id,neighbourhood_name,neighbourhood_district,room_type,accommodates,bathrooms,bedrooms,beds,amenities_list,price,minimum_nights,maximum_nights,has_availability,availability_30,availability_60,availability_90,availability_365,number_of_reviews,first_review_date,last_review_date,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,is_instant_bookable,reviews_per_month,country,city,insert_date
0,11964,A ROOM WITH A VIEW,Private bedroom in our attic apartment. Right ...,45553,Centro,None,Private room,2,2.0,2.0,1.0,"TV,Internet,Wifi,Air conditioning,Elevator,Buz...",400.0,3,365,True,7,20,40,130,78,2010-01-02,2010-01-02,97.0,100.0,100.0,100.0,100.0,100.0,100.0,False,75.0,spain,malaga,2018-07-31
1,21853,Bright and airy room,We have a quiet and sunny room with a good vie...,83531,C�rmenes,Latina,Private room,1,1.0,1.0,1.0,"TV,Internet,Wifi,Air conditioning,Kitchen,Free...",170.0,4,40,True,0,0,0,162,33,2014-10-10,2014-10-10,92.0,90.0,90.0,100.0,100.0,80.0,90.0,False,52.0,spain,madrid,2020-01-10
2,32347,Explore Cultural Sights from a Family-Friendly...,Open French doors and step onto a plant-filled...,139939,San Vicente,Casco Antiguo,Entire home/apt,4,1.0,1.0,2.0,"TV,Internet,Wifi,Air conditioning,Wheelchair a...",990.0,2,120,True,26,31,31,270,148,2011-01-05,2011-01-05,98.0,100.0,100.0,100.0,100.0,100.0,100.0,True,142.0,spain,sevilla,2019-07-29
3,35379,Double 02 CasanovaRooms Barcelona,Room at a my apartment. Kitchen and 2 bathroom...,152232,l'Antiga Esquerra de l'Eixample,Eixample,Private room,2,2.0,2.0,1.0,"TV,Internet,Wifi,Kitchen,Breakfast,Elevator,Bu...",400.0,2,730,True,9,23,49,300,292,2012-03-13,2012-03-13,94.0,100.0,90.0,100.0,100.0,100.0,90.0,True,306.0,spain,barcelona,2020-01-10
4,35801,Can Torras Farmhouse Studio Suite,Lay in bed & watch sunlight change the mood of...,153805,Quart,None,Private room,5,1.0,1.0,5.0,"Wifi,Pool,Free parking on premises,Breakfast,P...",900.0,1,180,True,0,19,49,312,36,2011-07-08,2011-07-08,97.0,100.0,100.0,100.0,100.0,100.0,100.0,False,39.0,spain,girona,2019-02-19
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6728,27237828,Studio at Finca Els Olivers,"Independent guest house, forming part of an ex...",1323233,B�scara,None,Entire home/apt,2,1.0,1.0,1.0,"TV,Cable TV,Internet,Wifi,Air conditioning,Poo...",1500.0,2,1124,True,22,47,77,78,1,2018-08-26,2018-08-26,100.0,100.0,100.0,100.0,100.0,100.0,100.0,False,10.0,spain,girona,2018-08-30
6729,27241318,ES MOLI D'EN SION - Villa with private pool in...,Enjoy the peace of the countryside in this bea...,80839530,Sa Pobla,None,Entire home/apt,10,4.0,4.0,7.0,"TV,Cable TV,Internet,Wifi,Air conditioning,Poo...",3130.0,7,1125,True,26,37,37,243,1,2020-03-13,2020-03-13,100.0,100.0,100.0,100.0,100.0,100.0,100.0,True,7.0,spain,mallorca,2020-04-23
6730,27244243,101.108_New building apartment with two double...,Apartment in Cadaqu�s center. 1rst �floor. Ele...,151496825,Cadaqu�s,None,Entire home/apt,4,1.0,1.0,2.0,"TV,Internet,Wifi,Air conditioning,Kitchen,Elev...",990.0,1,1125,True,24,40,40,40,0,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True,NaN,spain,girona,2018-08-30
6731,27244794,101.38_Apartment with one doble bedroom and te...,101.38.- Apartment placed Sa T�rtora � Sant An...,151496825,Cadaqu�s,None,Entire home/apt,2,1.0,1.0,2.0,"TV,Kitchen,Buzzer/wireless intercom,Family/kid...",720.0,1,1125,True,0,0,0,0,1,2018-09-23,2018-09-23,100.0,100.0,100.0,100.0,100.0,100.0,100.0,True,6.0,spain,girona,2019-12-31


## 02.- Filtrado solo variables necesarias

In [3]:
df_limpio.columns

Index(['apartment_id', 'name', 'description', 'host_id', 'neighbourhood_name', 'neighbourhood_district', 'room_type', 'accommodates', 'bathrooms', 'bedrooms',
       'beds', 'amenities_list', 'price', 'minimum_nights', 'maximum_nights', 'has_availability', 'availability_30', 'availability_60', 'availability_90',
       'availability_365', 'number_of_reviews', 'first_review_date', 'last_review_date', 'review_scores_rating', 'review_scores_accuracy',
       'review_scores_cleanliness', 'review_scores_checkin', 'review_scores_communication', 'review_scores_location', 'review_scores_value',
       'is_instant_bookable', 'reviews_per_month', 'country', 'city', 'insert_date'],
      dtype='object')

In [5]:
# Defino las variables que voy a usar para responder a la pregunta de negocio
columnas = [
    "apartment_id",
    "room_type",
    "availability_30",
    "availability_60",
    "availability_90",
    "availability_365",
    "city"   
]

# Creo un nuevo DF con solo las variables que voy a usar
df_operaciones = df_limpio[columnas].copy()
df_operaciones

,apartment_id,room_type,availability_30,availability_60,availability_90,availability_365,city
0,11964,Private room,7,20,40,130,malaga
1,21853,Private room,0,0,0,162,madrid
2,32347,Entire home/apt,26,31,31,270,sevilla
3,35379,Private room,9,23,49,300,barcelona
4,35801,Private room,0,19,49,312,girona
...,...,...,...,...,...,...,...
6728,27237828,Entire home/apt,22,47,77,78,girona
6729,27241318,Entire home/apt,26,37,37,243,mallorca
6730,27244243,Entire home/apt,24,40,40,40,girona
6731,27244794,Entire home/apt,0,0,0,0,girona


## 03.- Distribuciones (histogramas)

In [ ]:
variables = ["availability_30", "availability_60", "availability_90", "availability_365"]

# Crear figura 2x2
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=variables
)

# Añadir histogramas + KDE
row, col = 1, 1
for var in variables:

    # Histograma
    fig.add_trace(
        go.Histogram(
            x=df_operaciones[var],
            nbinsx=40,
            marker_color='#2E74B5',
            opacity=0.75,
            name=f"Hist {var}",
            showlegend=False
        ),
        row=row, col=col
    )

    # KDE (curva de densidad)
    fig.add_trace(
        go.Histogram(
            x=df_operaciones[var],
            histnorm='probability density',
            cumulative_enabled=False,
            opacity=0,
            marker_color='red',
            name=f"KDE {var}",
            showlegend=False
        ),
        row=row, col=col
    )

    # Avanzar posición en la cuadrícula
    col += 1
    if col == 3:
        col = 1
        row += 1

# Layout
fig.update_layout(
    title='<b>Distribución de Disponibilidad</b>',
    title_x=0.5,
    width=1000,
    height=700,
    template='plotly_white'
)

fig.show()


**NO tienen una distribucion normal.** Esto es relevante: ante distribuciones asimétricas, la **mediana** representa mejor el centro que la media, y conviene usar correlaciones de rango (Spearman).

In [ ]:
variables = ['availability_30', 'availability_60', 'availability_90', 'availability_365']
colores = ['#2E74B5', '#FF5733', '#28A745', '#8E44AD']

fig = go.Figure()

for i, var in enumerate(variables):
    fig.add_trace(go.Box(
        y=df_operaciones[var],
        name=var,
        boxpoints='outliers',
        marker_color=colores[i]   # ← ahora sí funciona
    ))

fig.update_layout(
    title='<b>Disponibilidad</b>',
    title_x=0.5,
    yaxis_title='Días',
    width=1200,
    height=500,
    template='plotly_white',
    showlegend=False      # ← oculta la leyenda
)

fig.show()


Se observa que, 

## 04.- Operaciones

### 04.1.- Media disponibilidad por ciudad

In [11]:
disp_final = df_operaciones.groupby('city')[['availability_30','availability_60',
                                  'availability_90','availability_365']].mean().round(2)
disp_final

,availability_30,availability_60,availability_90,availability_365
city,,,,
barcelona,11.11,25.63,42.50,182.09
girona,14.59,31.27,48.35,195.07
madrid,10.41,24.12,39.95,163.56
malaga,12.14,28.43,47.13,202.65
mallorca,13.41,28.70,45.17,210.92
menorca,14.99,30.88,47.54,199.49
sevilla,14.06,30.92,50.05,199.75
valencia,13.59,29.73,47.72,183.75


In [12]:

fig = go.Figure()

for col in disp_final.columns:
    fig.add_trace(go.Bar(
        x=disp_final.index,
        y=disp_final[col],
        name=col
    ))

fig.update_layout(
    title='<b>Disponibilidad media por ciudad y horizontes</b>',
    title_x=0.5,
    xaxis_title='Ciudad',
    yaxis_title='Días disponibles (media)',
    width=1300,
    height=500,
    template='plotly_white'
)

fig.show()


In [13]:


fig = go.Figure(data=go.Heatmap(
    z=disp_final.values,
    x=disp_final.columns,
    y=disp_final.index,
    colorscale='Blues',
    colorbar_title='Días'
))

fig.update_layout(
    title='<b>Disponibilidad media por ciudad y horizonte</b>',
    title_x=0.5,
    width=900,
    height=500,
    template='plotly_white'
)

fig.show()


### 04.2.- Media disponibilidad global

In [16]:
media_disp_glob = df_operaciones[['availability_30', 'availability_60', 'availability_90', 'availability_365']].mean().reset_index(name= 'Media_dispo_global_dias').round(2)

media_disp_glob 

,index,Media_dispo_global_dias
0,availability_30,12.29
1,availability_60,27.42
2,availability_90,44.30
3,availability_365,187.39


In [17]:


fig = go.Figure()

fig.add_trace(go.Scatter(
    x=media_disp_glob['index'],
    y=media_disp_glob['Media_dispo_global_dias'],
    mode='lines+markers',
    line=dict(color='#2E74B5', width=3),
    marker=dict(size=12, color='#FF5733'),
    showlegend=False
))

fig.update_layout(
    title='<b>Disponibilidad media global por horizonte</b>',
    title_x=0.5,
    xaxis_title='Horizonte',
    yaxis_title='Días disponibles (media)',
    template='plotly_white',
    width=900,
    height=450
)

fig.show()


In [18]:


# Horizontes a graficar
horizontes = ["availability_30", "availability_60", "availability_90", "availability_365"]

# Crear figura 2x2
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=horizontes
)

# Crear cada heatmap
row, col = 1, 1
for h in horizontes:

    # Tabla pivote: filas = city, columnas = room_type, valores = media
    tabla = df_operaciones.pivot_table(
        index="city",
        columns="room_type",
        values=h,
        aggfunc="mean"
    )

    fig.add_trace(
        go.Heatmap(
            z=tabla.values,
            x=tabla.columns,
            y=tabla.index,
            colorscale="Blues",
            colorbar_title="Días",
            showscale=True if (row == 1 and col == 1) else False  # solo un colorbar
        ),
        row=row, col=col
    )

    # Avanzar posición
    col += 1
    if col == 3:
        col = 1
        row += 1

# Layout general
fig.update_layout(
    title="<b>Disponibilidad media por ciudad y tipo de habitación</b>",
    title_x=0.5,
    width=1100,
    height=900,
    template="plotly_white"
)

fig.show()


In [19]:

horizontes = ["availability_30", "availability_60", "availability_90", "availability_365"]

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=horizontes
)

row, col = 1, 1
for h in horizontes:

    tabla = df_operaciones.pivot_table(
        index="city",
        columns="room_type",
        values=h,
        aggfunc="mean"
    )

    # HEATMAP
    fig.add_trace(
        go.Heatmap(
            z=tabla.values,
            x=tabla.columns,
            y=tabla.index,
            colorscale="Blues",
            showscale=True if (row == 1 and col == 1) else False
        ),
        row=row, col=col
    )

    # ANOTACIONES (texto dentro de cada celda)
    fig.add_trace(
        go.Scatter(
            x=np.repeat(tabla.columns, len(tabla.index)),
            y=np.tile(tabla.index, len(tabla.columns)),
            text=[f"{v:.1f}" for v in tabla.values.flatten()],
            mode="text",
            textfont=dict(color="black", size=12),
            showlegend=False
        ),
        row=row, col=col
    )

    col += 1
    if col == 3:
        col = 1
        row += 1

fig.update_layout(
    title="<b>Disponibilidad media por ciudad y tipo de habitación</b>",
    title_x=0.5,
    width=1100,
    height=900,
    template="plotly_white"
)

fig.show()


## 05.- Respuestas metodológicas (perfil Operaciones e Inventario)

### ¿Por qué esta técnica de análisis y no otra?
Para responder *"¿cuál es la disponibilidad media por ciudad y horizonte?"* he usado **estadística descriptiva agrupada** (medias y medianas por `city` y por ventana de disponibilidad) apoyada en **EDA visual** (histogramas, boxplots).

- **Tipo de datos:** observacionales, transversales (una *foto*), con variables numéricas (disponibilidad, precio) y categóricas (ciudad, tipo). No hay variable temporal continua fiable: `insert_date` es la fecha de extracción, no una serie de ocupación a lo largo del tiempo.
- **Objetivo:** *describir y comparar* el estado del inventario entre ciudades, no predecir ni inferir relaciones causales.
- **Ventajas en este contexto:** la estadística descriptiva agrupada es directa, interpretable por negocio y suficiente para comparar el inventario disponible. Una técnica predictiva (regresión, clustering) sería excesiva y poco fiable con datos estáticos para el objetivo planteado.

### ¿Qué supuestos tiene y cómo verifiqué que se cumplían?
- **Independencia de las observaciones:** se incumplía por los duplicados (el mismo alojamiento aparecía varias veces en distintas extracciones). Lo verifiqué con el conteo de `apartment_id` duplicados y lo **corregí deduplicando** (una fila por alojamiento).
- **Outliers y normalidad:** verifiqué con histogramas y boxplots que las distribuciones son **asimétricas y con outliers** (precio, reseñas). Por eso reporto **mediana** además de media y uso **Spearman** (que no exige normalidad ni linealidad) en lugar de Pearson.
- **Tamaño de muestra:** 7.001 registros (6.733 alojamientos únicos) es **amplio** para estimaciones descriptivas por ciudad; solo en ciudades con pocos anuncios (p. ej. Menorca) las medias son menos estables.
- **Calidad/escala:** verifiqué la escala real de `review_scores_rating` (0–1000) y que muchos ceros eran realmente faltantes (alojamientos sin reseñas), tratándolos como `NaN`.

### ¿Qué limitaciones tiene y cómo afectan a las conclusiones?
- **Foto estática:** no permite estudiar estacionalidad ni tendencias reales pese a que existan ventanas de 30/60/90/365 días; estas reflejan la previsión de disponibilidad en el momento de la extracción, no histórico.
- **Disponibilidad ≠ ocupación:** un día "no disponible" puede ser una reserva real o un bloqueo del host. Las conclusiones sobre demanda son **aproximaciones**, no medidas exactas de ocupación.
- **Permite concluir:** qué ciudades tienen, en promedio, más o menos inventario libre y cómo se distribuye por horizonte. **No permite concluir:** ocupación real, causalidad (p. ej. *por qué* una ciudad tiene baja disponibilidad), ni evolución temporal.
- **Mejoras futuras:** incorporar datos longitudinales (varias fotos por alojamiento con histórico de calendario), cruzar con reservas reales, normalizar la reputación por antigüedad (`reviews_per_month`) y, si se busca segmentar el inventario, aplicar *clustering* sobre disponibilidad y precio.


1. Lectura global: patrón clarísimo
Los cuatro heatmaps muestran un patrón muy consistente:

Cuanto mayor es el horizonte (30 → 365), mayor es la disponibilidad media.

Las islas (Mallorca, Menorca) y Málaga tienden a tener más disponibilidad en todos los horizontes.

Madrid y Barcelona muestran menor disponibilidad relativa, especialmente en horizontes cortos.

Los Hotel room suelen tener valores más altos que Private room o Shared room.

Esto ya te dice que la oferta turística se comporta de forma muy distinta según ciudad y tipo de alojamiento.

**2. Interpretación por horizontes:**

🟦 **availability_30** (corto plazo)
Las ciudades con mayor disponibilidad inmediata son:

Málaga, Sevilla, Mallorca, Menorca

Las ciudades con menor disponibilidad inmediata:

Madrid, Barcelona, Valencia

Los Hotel room destacan en casi todas las ciudades.

Los Shared room tienen valores muy dispares (desde 0 en Málaga hasta >15 en Girona/Barcelona).

👉 Conclusión: alta presión de demanda en grandes ciudades, más holgura en destinos turísticos costeros.


🟦 **availability_60** (medio plazo)
El patrón se amplifica:
Málaga, Sevilla, Mallorca y Menorca siguen arriba.

Madrid y Barcelona siguen siendo las más tensas.

Los Hotel room y Entire home/apt suben de forma notable.

👉 Conclusión: los destinos turísticos tienen más planificación anticipada, mientras que las grandes ciudades mantienen ocupación sostenida.


🟦 **availability_90** (trimestre)
Se ve un salto fuerte en todas las ciudades.

Málaga y Sevilla destacan especialmente.

En Madrid, los Private room suben mucho (57.5 días), señal de que este segmento es más elástico.

👉 Conclusión: a 90 días, el mercado se “abre”, pero las diferencias entre ciudades se mantienen.


🟦 availability_365 (anual)
Aquí es donde el heatmap explota visualmente:

Mallorca, Málaga, Girona y Sevilla tienen valores muy altos (180–275 días).

Madrid y Barcelona siguen siendo las más bajas.

Los Private room y Shared room muestran comportamientos muy distintos según ciudad:

Málaga Shared room = 0 días (muy llamativo)

Valencia Shared room = 222 días

Madrid Private room = 217 días

👉 Conclusión:
El mercado anual muestra enormes diferencias estructurales entre ciudades y tipos de alojamiento.
Las islas y Málaga tienen mucha más disponibilidad anual, mientras que Madrid y Barcelona tienen mercados más tensos y ocupados.

**3. Interpretación por tipo de habitación**

🏠 Entire home/apt
Más disponibilidad en Málaga, Sevilla, Mallorca, Girona.

Menos en Madrid y Barcelona.

🏨 Hotel room
Muy consistente: siempre valores altos.

Especialmente fuerte en Menorca, Málaga, Mallorca.

🚪 Private room
Mucha variabilidad.

En Madrid y Barcelona sube mucho en horizontes largos.

🛏️ Shared room
Extremadamente desigual:

Málaga = 0 días en varios horizontes

Valencia = valores muy altos

Barcelona = valores medios-altos

👉 Esto indica que el mercado de shared rooms es muy dependiente de la ciudad.

**4. Conclusiones estratégicas**

✔ Las islas y Málaga tienen la mayor disponibilidad estructural
Esto sugiere mercados más estacionales y con mayor oferta.

✔ Madrid y Barcelona son mercados tensos
Incluso a 365 días, la disponibilidad es baja comparada con otras ciudades.

✔ Los hoteles son el tipo más estable
Siempre aparecen con valores altos y consistentes.

✔ Shared room es el segmento más volátil
Puede ser casi nulo (Málaga) o muy alto (Valencia).

### Rankings

In [20]:
# Ranking general por ciudad
ranking_general = (
    df_operaciones.groupby("city")[["availability_30","availability_60","availability_90","availability_365"]]
    .mean()
    .mean(axis=1)
    .sort_values(ascending=False)
)

print("Ranking general por ciudad:")
print(ranking_general)

# Ranking por horizonte
ranking_por_horizonte = (
    df_operaciones.groupby("city")[["availability_30","availability_60","availability_90","availability_365"]]
    .mean()
    .sort_values(by=["availability_30","availability_60","availability_90","availability_365"], ascending=False)
)

print("\nRanking por horizonte:")
print(ranking_por_horizonte)


Ranking general por ciudad:
city
mallorca     74.547217
sevilla      73.695015
menorca      73.224638
malaga       72.587758
girona       72.320737
valencia     68.697811
barcelona    65.334272
madrid       59.512357
dtype: float64

Ranking por horizonte:
           availability_30  availability_60  availability_90  availability_365
city                                                                          
menorca          14.985507        30.876812        47.543478        199.492754
girona           14.587097        31.272811        48.352074        195.070968
sevilla          14.058651        30.920821        50.049853        199.750733
valencia         13.589226        29.730640        47.717172        183.754209
mallorca         13.407847        28.700730        45.165146        210.915146
malaga           12.144543        28.430678        47.126844        202.648968
barcelona        11.114160        25.631553        42.503675        182.087702
madrid           10.414756       

In [21]:
ranking_general = (
    df_operaciones
    .groupby("city")[["availability_30","availability_60","availability_90","availability_365"]]
    .mean()
    .mean(axis=1)
    .sort_values(ascending=False)
    .round(1)
    .reset_index(name="Media_total_dias")
)

ranking_general


,city,Media_total_dias
0,mallorca,74.5
1,sevilla,73.7
2,menorca,73.2
3,malaga,72.6
4,girona,72.3
5,valencia,68.7
6,barcelona,65.3
7,madrid,59.5


La interpretación cualitativa:

 - Destinos turísticos como Mallorca, Sevilla, Menorca, Málaga, Girona → más días disponibles de media.

 - Grandes ciudades como Madrid y Barcelona → menos disponibilidad media, mercados más tensos.